In [ ]:
!pip install -q transformers accelerate bitsandbytes peft trl datasets scikit-learn pynvml gdown

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd

from PIL import Image

from tqdm import tqdm

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report,

    confusion_matrix
)

from transformers import (

    LlavaForConditionalGeneration,

    AutoProcessor
)

In [ ]:
import gdown

folder_url = "https://drive.google.com/drive/folders/1hKLOtpVmF45IoBmJPwojgq6XraLtHmV6"

gdown.download_folder(

    url=folder_url,

    output="MultiOFF",

    quiet=False,

    remaining_ok=True
)

In [ ]:
for root, dirs, files in os.walk("./MultiOFF"):

    print(root)

    print(files[:5])

    print("=" * 50)

In [ ]:
DATASET_DIR = "./MultiOFF"

TRAIN_FILE = os.path.join(

    DATASET_DIR,

    "Split Dataset",

    "Training_meme_dataset.csv"
)

DEV_FILE = os.path.join(

    DATASET_DIR,

    "Split Dataset",

    "Validation_meme_dataset.csv"
)

TEST_FILE = os.path.join(

    DATASET_DIR,

    "Split Dataset",

    "Testing_meme_dataset.csv"
)

IMAGE_DIR = os.path.join(

    DATASET_DIR,

    "Labelled Images"
)

In [ ]:
train_df = pd.read_csv(TRAIN_FILE)

dev_df = pd.read_csv(DEV_FILE)

test_df = pd.read_csv(TEST_FILE)

In [ ]:
print("TRAIN:", len(train_df))
print("DEV:", len(dev_df))
print("TEST:", len(test_df))

In [ ]:
print(train_df.columns)

In [ ]:
train_df = train_df.rename(columns={
    "image_name": "img",
    "sentence": "text"
})

dev_df = dev_df.rename(columns={
    "image_name": "img",
    "sentence": "text"
})

test_df = test_df.rename(columns={
    "image_name": "img",
    "sentence": "text"
})


In [ ]:
print(train_df.head())

In [ ]:
# ============================================================
# STEP 12 — CLEAN LABELS
# ============================================================

print(train_df["label"].unique())

print(dev_df["label"].unique())

print(test_df["label"].unique())

In [ ]:
train_df["label"] = (

    train_df["label"]

    .astype(str)

    .str.strip()
)

dev_df["label"] = (

    dev_df["label"]

    .astype(str)

    .str.strip()
)

test_df["label"] = (

    test_df["label"]

    .astype(str)

    .str.strip()
)

In [ ]:
label_map = {

    "offensive": 1,

    "Non-offensiv": 0,

    "non-offensiv": 0,

    "non-offensive": 0,

    "OFF": 1,

    "NOT": 0
}

In [ ]:
train_df["label"] = train_df["label"].map(label_map)

dev_df["label"] = dev_df["label"].map(label_map)

test_df["label"] = test_df["label"].map(label_map)

In [ ]:
train_df = train_df.dropna(subset=["label"])

dev_df = dev_df.dropna(subset=["label"])

test_df = test_df.dropna(subset=["label"])

In [ ]:
train_df["label"] = train_df["label"].astype(int)

dev_df["label"] = dev_df["label"].astype(int)

test_df["label"] = test_df["label"].astype(int)

In [ ]:
print(train_df["label"].unique())

print(dev_df["label"].unique())

print(test_df["label"].unique())

In [ ]:
def filter_existing_images(df):

    valid_rows = []

    for idx in range(len(df)):

        row = df.iloc[idx]

        image_path = os.path.join(
            IMAGE_DIR,
            row["img"]
        )

        if os.path.exists(image_path):

            valid_rows.append(row)

    return pd.DataFrame(valid_rows)
train_df = filter_existing_images(train_df)

dev_df = filter_existing_images(dev_df)

test_df = filter_existing_images(test_df)

In [ ]:
print("TRAIN:", len(train_df))
print("DEV:", len(dev_df))
print("TEST:", len(test_df))

In [ ]:
train_df = train_df.head(100)

dev_df = dev_df.head(20)

test_df = test_df.head(20)

In [ ]:
train_data = train_df.to_dict("records")

dev_data = dev_df.to_dict("records")

test_data = test_df.to_dict("records")

In [ ]:
sample = train_data[0]

print(sample)

In [ ]:
image_path = os.path.join(
    IMAGE_DIR,
    sample["img"]
)

image = Image.open(image_path).convert("RGB")

print(image)

In [ ]:
def convert_to_llava_format(data):

    converted = []

    for sample in data:

        image_path = os.path.join(
            IMAGE_DIR,
            sample["img"]
        )

        converted.append({

            "image": image_path,

            "text": sample["text"],

            "label": sample["label"]
        })

    return converted

In [ ]:
train_converted = convert_to_llava_format(train_data)

dev_converted = convert_to_llava_format(dev_data)

test_converted = convert_to_llava_format(test_data)

In [ ]:
import os
import time
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from tqdm import tqdm

from datasets import Dataset

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report,

    confusion_matrix,

    roc_curve,

    auc
)

from transformers import (

    LlavaForConditionalGeneration,

    AutoProcessor,

    TrainingArguments,

    Trainer,

    BitsAndBytesConfig
)

from peft import (

    prepare_model_for_kbit_training,

    LoraConfig,

    get_peft_model
)

train_dataset = Dataset.from_list(train_converted)

dev_dataset = Dataset.from_list(dev_converted)

test_dataset = Dataset.from_list(test_converted)

In [ ]:
print(train_dataset)

In [ ]:
MODEL_ID = "llava-hf/llava-1.5-7b-hf"

In [ ]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

In [ ]:
model = LlavaForConditionalGeneration.from_pretrained(

    MODEL_ID,

    quantization_config=bnb_config,

    device_map="auto"
)

In [ ]:
model = prepare_model_for_kbit_training(
    model
)

In [ ]:
lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"
)

In [ ]:
model = get_peft_model(
    model,
    lora_config
)

In [ ]:
model.print_trainable_parameters()

In [ ]:
def preprocess_function(example):

    image = Image.open(
        example["image"]
    ).convert("RGB")

    label_text = (
        "Yes"
        if example["label"] == 1
        else "No"
    )

    prompt = f"""
USER:
<image>

Meme Text:
{example['text']}

Is this meme offensive or hateful?

Answer only Yes or No.

ASSISTANT:
{label_text}
"""

    inputs = processor(

        text=prompt,

        images=image,

        return_tensors="pt",

        padding=True

    )

    inputs = {

        k: v.squeeze(0)

        for k, v in inputs.items()
    }

    inputs["labels"] = inputs["input_ids"].clone()

    return inputs

In [ ]:
train_dataset = train_dataset.map(
    preprocess_function
)
dev_dataset = dev_dataset.map(
    preprocess_function
)

In [ ]:
columns_to_keep = [
    "input_ids",
    "attention_mask",
    "pixel_values",
    "labels"
]
train_dataset = train_dataset.remove_columns(
    [
        col for col in train_dataset.column_names
        if col not in columns_to_keep
    ]
)
dev_dataset = dev_dataset.remove_columns(
    [
        col for col in dev_dataset.column_names
        if col not in columns_to_keep
    ]
)

In [ ]:
print(train_dataset[0].keys())

In [ ]:
training_args = TrainingArguments(

    output_dir="./llava_multioff_output",

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=4,

    num_train_epochs=20,

    learning_rate=2e-4,

    logging_steps=1,

    save_strategy="epoch",

    eval_strategy="epoch",

    fp16=True,

    remove_unused_columns=False,

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=dev_dataset
)

In [ ]:
start_train_time = time.time()
trainer.train()
end_train_time = time.time()
training_time = (
    end_train_time - start_train_time
)
print(training_time)

In [ ]:
#trainer.save_model(
   # "./llava_multioff_model"
#)

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
def predict_llava(image_path, text):

    image = Image.open(
        image_path
    ).convert("RGB")

    prompt = f"""
USER:
<image>

Meme Text:
{text}

Is this meme offensive or hateful?

Answer only Yes or No.

ASSISTANT:
"""

    inputs = processor(

        text=prompt,

        images=image,

        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=10
        )

    result = processor.batch_decode(

        outputs,

        skip_special_tokens=True
    )[0]

    result = result.lower()

    if "yes" in result:

        return 1

    else:

        return 0

In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()

if GPU_AVAILABLE:

    try:

        from pynvml import *

        nvmlInit()

        handle = nvmlDeviceGetHandleByIndex(0)

        ENERGY_MODE = "GPU"

    except:

        ENERGY_MODE = "CPU"

else:

    ENERGY_MODE = "CPU"

In [ ]:
def get_power_usage():

    if ENERGY_MODE == "GPU":

        return (
            nvmlDeviceGetPowerUsage(handle)
            / 1000
        )

    else:

        return 65

In [ ]:
def compute_energy_kwh(
    power_watts,
    duration_seconds
):

    return (
        power_watts * duration_seconds
    ) / (1000 * 3600)

In [ ]:
y_true = []

y_pred = []

y_scores = []

inference_times = []

energy_consumptions = []

In [ ]:
for sample in tqdm(test_data):

    image_path = os.path.join(
        IMAGE_DIR,
        sample["img"]
    )

    text = sample["text"]

    label = sample["label"]

    start_time = time.time()

    power = get_power_usage()

    prediction = predict_llava(
        image_path,
        text
    )

    end_time = time.time()

    duration = (
        end_time - start_time
    )

    energy = compute_energy_kwh(
        power,
        duration
    )

    y_true.append(label)

    y_pred.append(prediction)

    y_scores.append(prediction)

    inference_times.append(duration)

    energy_consumptions.append(energy)

In [ ]:
print(set(y_true))

In [ ]:
accuracy = accuracy_score(
    y_true,
    y_pred
)
precision = precision_score(
    y_true,
    y_pred
)
recall = recall_score(
    y_true,
    y_pred
)
f1 = f1_score(
    y_true,
    y_pred
)

In [ ]:
avg_time = np.mean(
    inference_times
)
avg_energy = np.mean(
    energy_consumptions
)

In [ ]:
print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1-score:", f1)

print("Inference Time:", avg_time)

print("Energy (kWh):", avg_energy)

In [ ]:
report = classification_report(

    y_true,

    y_pred,

    target_names=[
        "Non-Offensive",
        "Offensive"
    ]
)
print(report)